In [ ]:
import pandas as pd
import pickle
import importlib
import sys
import os
from syllabification import tokenize, syllabify_sentences
from markov_models import MarkovModel 
from helpers import get_word_data, get_info_rate, update_values_in_csv
import warnings
import re
import numpy as np

# Define parameters
language = "YUE"
input_type = "words"

language_paths_words = {
    "FRA": "Z:/data/FRA/Lexique383.tsv",
    "JPN": "Z:/data/JPN/jpn.txt",
    "CMN": "Z:/data/CMN/cmn.txt",
    "VIE": "Z:/data/VIE/vie.txt",
    "YUE": "Z:/data/YUE/yue.txt", 
    "ENG": "Z:/data/ENG/eng.txt",
    "DEU": "Z:/data/DEU/WebCelex_German.xlsx",
}

n_values = [1, 2, 3, 4]  # For bigram, trigram, and 4-gram models
markov_models = {}

for n in n_values:
    
    # Create and build the Markov model
    model = MarkovModel(n)

    if input_type == "words": 
        try:
            path = language_paths_words[language]
        except KeyError:
            raise ValueError(f"Unsupported language: {language}")

        # Load the data
        words = get_word_data(path)
       
        print(f"\nTraining a Markov Model with n = {n}:")
        # Build the markov model
        model.build(words, input_type)
    
    elif input_type == "sentences" and language == "FRA": # possible only for FRA
        # Load the paired data
        with open(f"produced_data/{language}/sentence_pairs.pkl", "rb") as f: 
            sentence_pairs = pickle.load(f)

        # Merge all transcribed (syllabified) sentences into one list
        merged_sentences = []
        for tokenized, transcribed in sentence_pairs:
            merged_sentences.extend(transcribed)

        model.build(merged_sentences, input_type)
    else: raise ValueError(f"For {language}, sentence data is not available, please set input_type=words")


    # Compute the conditional entropy (information density)
    info_density = model.compute_conditional_entropy()
    print(f"Information Density: {info_density:.4f}")

    # Compute the information rate (bits per second)
    info_rate = get_info_rate(info_density, language)
    print(f"Information Rate: {np.mean(info_rate):.4f}")
    
    # Update the CSV file with the computed info_density and info_rate
    update_values_in_csv(language, info_density, n, 'ID')
    update_values_in_csv(language, info_rate, n, 'IR')

    # Store model for later use 
    markov_models[n] = model

    # Display exactly 3 examples
    example_count = 0
    print("\nExample probabilities (p(x, y)):")

    for (prefix, suffix), p_xy in model.cond_probs.items():
        print(f"p({prefix} -> {suffix}) = {p_xy:.4f}")
        example_count += 1
        if example_count == 3:
            break
    
    # Save the model to a file
    model.save_model(language, input_type)

Language: VIE

Training a Markov Model with n = 1:
ngram examples: [('xoŋ͡m1',), ('xoŋ͡m1',), ('xoŋ͡m1',), ('xoŋ͡m1',), ('xoŋ͡m1',)]
Information Density: 9.7205
Information Rate: 49.4458
Updated ID_unigram_esidaine
Updated IR_unigram_esidaine

Example probabilities (p(x, y)):
p(() -> xoŋ͡m1) = 0.0148
p(() -> mot6) = 0.0145
p(() -> kuə4) = 0.0119

✅ Saved 1-gram model to 'produced_data/VIE/'
Language: VIE

Training a Markov Model with n = 2:
ngram examples: [('kɔ5', 'tʰe4'), ('kɔ5', 'tʰe4'), ('kɔ5', 'tʰe4'), ('kɔ5', 'tʰe4'), ('kɔ5', 'tʰe4')]
Information Density: 2.1713
Information Rate: 11.0446
Updated ID_bigram_esidaine
Updated IR_bigram_esidaine

Example probabilities (p(x, y)):
p(('kɔ5',) -> tʰe4) = 0.5359
p(('cɔ1',) -> biət5) = 0.6046
p(('sɯ4',) -> zuŋ͡m6) = 0.6994

✅ Saved 2-gram model to 'produced_data/VIE/'
Language: VIE

Training a Markov Model with n = 3:
ngram examples: [('toŋ͡m4', 'koŋ͡m1', 'ti1'), ('toŋ͡m4', 'koŋ͡m1', 'ti1'), ('toŋ͡m4', 'koŋ͡m1', 'ti1'), ('toŋ͡m4', 'koŋ͡m1',